# 03 — Preprocessing
Student Success Analytics Platform

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.database import run_query
from src.preprocessing import (
    build_feature_matrix, get_column_types, split_data,
    fit_transformers, transform_features, save_transformers
)

print("libraries loaded")

libraries loaded


In [2]:
# load from SQLite (notebook 2's output) rather than re-reading the CSV — confirms the SQL layer is actually load-bearing
df = run_query("SELECT * FROM students;")
print("shape:", df.shape)

shape: (80000, 35)


In [3]:
# build feature matrix + both targets; grade_tier, exam_score, dropout_risk, student_id excluded from X via NON_FEATURE_COLS
X, y_reg, y_clf = build_feature_matrix(df)
print("X shape:", X.shape)
print("y_reg (exam_score) sample:", y_reg.head().tolist())
print("y_clf (dropout_risk) value counts:\n", y_clf.value_counts())

X shape: (80000, 31)
y_reg (exam_score) sample: [100, 99, 98, 100, 98]
y_clf (dropout_risk) value counts:
 dropout_risk
0    78418
1     1582
Name: count, dtype: int64


In [4]:
numeric_cols, categorical_cols = get_column_types(X)
print(f"numeric features: {len(numeric_cols)}")
print(numeric_cols)
print(f"\ncategorical features: {len(categorical_cols)}")
print(categorical_cols)

# cardinality check before one-hot encoding — flags any column that would blow up dimensionality
print("\ncardinality per categorical column:")
print(X[categorical_cols].nunique())

numeric features: 20
['age', 'study_hours_per_day', 'social_media_hours', 'netflix_hours', 'attendance_percentage', 'sleep_hours', 'exercise_frequency', 'mental_health_rating', 'previous_gpa', 'semester', 'stress_level', 'social_activity', 'screen_time', 'parental_support_level', 'motivation_level', 'exam_anxiety_score', 'time_management_score', 'wellness_score', 'distraction_hours', 'study_efficiency']

categorical features: 11
['gender', 'major', 'part_time_job', 'diet_quality', 'parental_education_level', 'internet_quality', 'extracurricular_participation', 'study_environment', 'access_to_tutoring', 'family_income_range', 'learning_style']

cardinality per categorical column:
gender                           3
major                            6
part_time_job                    2
diet_quality                     3
parental_education_level         5
internet_quality                 3
extracurricular_participation    2
study_environment                5
access_to_tutoring              

## Train/test split
Stratified on `dropout_risk` (`y_clf`) since it's severely imbalanced (~2% positive) — a plain random split risks too few positive cases landing in the test set to evaluate meaningfully. `exam_score` doesn't need its own stratification; it shares the same split indices as `y_clf`, so both targets stay aligned to the same train/test student rows.

In [5]:
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = split_data(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

print("train shape:", X_train.shape, " test shape:", X_test.shape)
print("dropout rate — train:", y_clf_train.mean().round(4), " test:", y_clf_test.mean().round(4))

train shape: (64000, 31)  test shape: (16000, 31)
dropout rate — train: 0.0198  test: 0.0198


## Encoding + scaling
Fit `OneHotEncoder` and `StandardScaler` on the **training set only**, then transform both train and test. Fitting on test data (or on the full dataset before splitting) leaks the test distribution into training and inflates evaluation metrics — a subtler version of the same leakage problem as `grade_tier` in notebook 1.

In [6]:
encoder, scaler = fit_transformers(X_train, numeric_cols, categorical_cols)

X_train_processed = transform_features(X_train, encoder, scaler, numeric_cols, categorical_cols)
X_test_processed = transform_features(X_test, encoder, scaler, numeric_cols, categorical_cols)

print("processed train shape:", X_train_processed.shape)
print("processed test shape:", X_test_processed.shape)

processed train shape: (64000, 58)
processed test shape: (16000, 58)


## SMOTE — classification training set only
Applied strictly **after** the split and strictly to **training data**. Never apply before splitting or to the test set: synthetic samples derived from test-set neighbors would leak test information into training, and evaluating on synthetic data would give a falsely optimistic read on real-world performance. This resampled set is for the classification model only — the regression target (`exam_score`) is untouched.

In [7]:
from imblearn.over_sampling import SMOTE

print("before SMOTE:", y_clf_train.value_counts().to_dict())

smote = SMOTE(random_state=42)
X_train_resampled, y_clf_train_resampled = smote.fit_resample(X_train_processed, y_clf_train)

print("after SMOTE: ", y_clf_train_resampled.value_counts().to_dict())

before SMOTE: {0: 62734, 1: 1266}
after SMOTE:  {0: 62734, 1: 62734}


In [8]:
import os
os.makedirs("../models", exist_ok=True)

save_transformers(encoder, scaler, out_dir="../models")

X_train_processed.to_csv("../data/processed/X_train_processed.csv", index=False)
X_test_processed.to_csv("../data/processed/X_test_processed.csv", index=False)
X_train_resampled.to_csv("../data/processed/X_train_resampled.csv", index=False)

y_reg_train.to_csv("../data/processed/y_reg_train.csv", index=False)
y_reg_test.to_csv("../data/processed/y_reg_test.csv", index=False)
y_clf_train.to_csv("../data/processed/y_clf_train.csv", index=False)
y_clf_test.to_csv("../data/processed/y_clf_test.csv", index=False)
y_clf_train_resampled.to_csv("../data/processed/y_clf_train_resampled.csv", index=False)

print("encoder + scaler saved to ../models/")
print("processed splits saved to ../data/processed/")

encoder + scaler saved to ../models/
processed splits saved to ../data/processed/


In [9]:
print("=" * 50)
print("  PREPROCESSING SUMMARY")
print("=" * 50)
print(f"  train rows: {X_train_processed.shape[0]:,}  (resampled for clf: {X_train_resampled.shape[0]:,})")
print(f"  test rows:  {X_test_processed.shape[0]:,}")
print(f"  features after encoding: {X_train_processed.shape[1]}")
print(f"  dropout rate train/test: {y_clf_train.mean():.4f} / {y_clf_test.mean():.4f}")
print("=" * 50)
print()
print("next: 04_feature_engineering.ipynb")

  PREPROCESSING SUMMARY
  train rows: 64,000  (resampled for clf: 125,468)
  test rows:  16,000
  features after encoding: 58
  dropout rate train/test: 0.0198 / 0.0198

next: 04_feature_engineering.ipynb
